# Benchmark: Gators vs Feature-Engine Datetime Features

This notebook compares the performance of **Gators datetime feature generation** (Polars-based) vs **feature-engine DatetimeFeatures** (pandas-based) across different dataset sizes.

**Objective**: Demonstrate Gators' multi-core processing advantages for datetime feature extraction tasks.

**Transformers Compared**:
- `gators.feature_generation_dt.OrdinalFeatures` vs `feature_engine.datetime.DatetimeFeatures`
- `gators.feature_generation_dt.CyclicFeatures` vs `feature_engine.datetime.DatetimeFeatures` (with cyclical features)

**Dataset Sizes**: 1K, 10K, 100K, 1M rows

**Note**: Feature-engine operates on pandas DataFrames, while Gators uses Polars for multi-core parallel processing.

## Setup & Imports

In [9]:
import polars as pl
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Gators datetime feature generators
from gators.feature_generation_dt import (
    OrdinalFeatures,
    CyclicFeatures
)

# Feature-engine datetime features
from feature_engine.datetime import DatetimeFeatures

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

print("✅ Imports complete")
print(f"Feature-engine version: {__import__('feature_engine').__version__}")

✅ Imports complete
Feature-engine version: 1.9.4


## Utility Functions

In [10]:
# Import benchmark utilities
from benchmark import (
    generate_datetime_data,
    benchmark_transformer,
    DATASET_SIZES,
    DEFAULT_TIMEOUT
)

print("✅ Benchmark utilities imported from benchmark.py")

✅ Benchmark utilities imported from benchmark.py


## Generate Test Datasets

In [11]:
# Use dataset sizes from constants
dataset_sizes = DATASET_SIZES

# Generate datasets (deterministic - no randomness)
datasets = {}
for size in dataset_sizes:
    print(f"Generating {size:,} row dataset...")
    df_polars, df_pandas = generate_datetime_data(n_rows=size, n_datetime_cols=3)
    datasets[size] = {'polars': df_polars, 'pandas': df_pandas}
    print(f"  Polars: {df_polars.shape}, Pandas: {df_pandas.shape}")
    print(f"  Columns: {list(df_polars.columns)}")

print("\n✅ All datasets generated (deterministic - fully reproducible)")

Generating 1,000 row dataset...
  Polars: (1000, 3), Pandas: (1000, 3)
  Columns: ['datetime_0', 'datetime_1', 'datetime_2']
Generating 10,000 row dataset...
  Polars: (10000, 3), Pandas: (10000, 3)
  Columns: ['datetime_0', 'datetime_1', 'datetime_2']
Generating 100,000 row dataset...
  Polars: (100000, 3), Pandas: (100000, 3)
  Columns: ['datetime_0', 'datetime_1', 'datetime_2']
Generating 1,000,000 row dataset...
  Polars: (1000000, 3), Pandas: (1000000, 3)
  Columns: ['datetime_0', 'datetime_1', 'datetime_2']

✅ All datasets generated (deterministic - fully reproducible)


## Benchmark 1: Ordinal Features Extraction

Compare `gators.feature_generation_dt.OrdinalFeatures` vs `feature_engine.datetime.DatetimeFeatures`.

Extract year, month, day, hour, minute, second from datetime columns.

In [12]:
ordinal_results = []

# Get datetime column names
datetime_cols = list(datasets[1_000]['polars'].columns)

print(f"\n{'='*70}")
print(f"Testing Ordinal Features Extraction ({len(datetime_cols)} datetime columns)")
print(f"Timeout: {DEFAULT_TIMEOUT}s per operation")
print(f"{'='*70}")

for size in dataset_sizes:
    X_polars = datasets[size]['polars']
    X_pandas = datasets[size]['pandas']
    
    # Create transformers
    # Gators: Extract year, month, day, hour, minute, second
    gators_ord = OrdinalFeatures(
        subset=datetime_cols,
        components=['year', 'semester', 'quarter', 'month', 'week', 'day_of_week', 'day_of_month', 'day_of_year', 'weekend', 'leap_year', 'hour', 'minute', 'second']
    )
    
    # Feature-engine: Extract same features
    fe_ord = DatetimeFeatures(
        variables=datetime_cols,
        features_to_extract=['year', 'semester', 'quarter', 'month', 'week', 'day_of_week', 'day_of_month', 'day_of_year', 'weekend', 'leap_year', 'hour', 'minute', 'second']
    )
    
    # Benchmark with timeout
    try:
        results = benchmark_transformer(
            gators_ord,
            fe_ord,
            X_polars,
            X_pandas,
            n_runs=3,
            timeout_seconds=DEFAULT_TIMEOUT
        )
        
        # Store results
        ordinal_results.append({
            'feature_type': 'Ordinal',
            'dataset_size': size,
            'n_columns': len(datetime_cols),
            'gators_fit': results['gators_fit'],
            'gators_transform': results['gators_transform'],
            'gators_total': results['gators_total'],
            'fe_fit': results['comparison_fit'],
            'fe_transform': results['comparison_transform'],
            'fe_total': results['comparison_total'],
            'speedup_total': results['speedup_total']
        })
        
        print(f"  {size:>8,} rows: Gators={results['gators_total']:.4f}s, "
              f"Feature-engine={results['comparison_total']:.4f}s, "
              f"Speedup={results['speedup_total']:.2f}x")
    except Exception as e:
        print(f"  {size:>8,} rows: ⚠️  Error - {str(e)}")
        continue

# Convert to DataFrame for analysis
all_results = pd.DataFrame(ordinal_results)
print("\n✅ Ordinal features extraction benchmarks complete")


Testing Ordinal Features Extraction (3 datetime columns)
Timeout: 180s per operation
     1,000 rows: Gators=0.0010s, Feature-engine=0.0065s, Speedup=6.85x
    10,000 rows: Gators=0.0010s, Feature-engine=0.0197s, Speedup=19.57x
   100,000 rows: Gators=0.0037s, Feature-engine=0.0695s, Speedup=18.87x
  1,000,000 rows: Gators=0.0325s, Feature-engine=0.5197s, Speedup=15.99x

✅ Ordinal features extraction benchmarks complete


## Summary Statistics

In [13]:

# Summary by dataset size
summary_by_size = all_results.groupby('dataset_size').agg({
    'gators_total': 'mean',
    'fe_total': 'mean',
    'speedup_total': 'mean'
}).round(4)

summary_by_size.columns = ['Avg Gators Time (s)', 'Avg Feature-engine Time (s)', 'Avg Speedup']
summary_by_size.index = [f'{size:,} rows' for size in summary_by_size.index]

print("\n" + "="*80)
print("SUMMARY BY DATASET SIZE")
print("="*80)
print(summary_by_size.to_string())
print("\n" + "="*80)


SUMMARY BY DATASET SIZE
                Avg Gators Time (s)  Avg Feature-engine Time (s)  Avg Speedup
1,000 rows                   0.0010                       0.0065       6.8529
10,000 rows                  0.0010                       0.0197      19.5685
100,000 rows                 0.0037                       0.0695      18.8651
1,000,000 rows               0.0325                       0.5197      15.9905

